In [45]:
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import matplotlib.ticker as mtick
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

df2 = pd.read_csv('titanic_train.csv')

# -----------------------------
# 2. 결측치 처리
# -----------------------------
df2['Age'] = df2['Age'].fillna(df2['Age'].median())
df2['Embarked'] = df2['Embarked'].fillna(df2['Embarked'].mode()[0])
df2['Fare'] = df2['Fare'].fillna(df2['Fare'].median())

# Cabin은 결측이 많아서 존재 여부만 활용
df2['Cabin'] = df2['Cabin'].fillna('U')
df2['HasCabin'] = (df2['Cabin'] != 'U').astype(int)

# -----------------------------
# 3. 기본 범주형 처리
# -----------------------------
sex_mapping = {'male': 0, 'female': 1}
df2['Sex'] = df2['Sex'].map(sex_mapping)

# Embarked 원핫인코딩
df2 = pd.get_dummies(df2, columns=['Embarked'], drop_first=True)

# bool 타입이면 int로 변환
for col in df2.columns:
    if df2[col].dtype == 'bool':
        df2[col] = df2[col].astype(int)

# -----------------------------
# 4. Name에서 Title 추출
# -----------------------------
df2['Name_Length'] = df2['Name'].apply(len)
df2['LastName'] = df2['Name'].str.split(',').str[0]
df2['FamilyName_Count'] = df2.groupby('LastName')['LastName'].transform('count')

df2['Title'] = df2['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# 희귀 호칭 묶기
df2['Title'] = df2['Title'].replace([
    'Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
    'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'
], 'Rare')

df2['Title'] = df2['Title'].replace({
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'Mme': 'Mrs'
})

title_mapping = {
    'Mr': 0,
    'Miss': 1,
    'Mrs': 2,
    'Master': 3,
    'Rare': 4
}
df2['Title'] = df2['Title'].map(title_mapping)

# -----------------------------
# 5. Age 그룹화
# -----------------------------
bins = [0, 12, 19, 35, 60, 100]
labels = ['Child', 'Teenager', 'YoungAdult', 'Adult', 'Senior']
df2['Age_Group'] = pd.cut(df2['Age'], bins=bins, labels=labels, right=True, include_lowest=True)

agegroup_mapping = {
    'Child': 0,
    'Teenager': 1,
    'YoungAdult': 2,
    'Adult': 3,
    'Senior': 4
}
df2['Age_Group'] = df2['Age_Group'].map(agegroup_mapping)
scaler = StandardScaler()
df2['Age_scaled'] = scaler.fit_transform(df2[['Age']])
# -----------------------------
# 6. 가족 관련 파생변수
# -----------------------------
df2['FamilySize'] = df2['SibSp'] + df2['Parch'] + 1
df2['IsAlone'] = (df2['FamilySize'] == 1).astype(int)

# FamilySize 그룹화도 추가 가능
def family_label(size):
    if size == 1:
        return 0      # Alone
    elif 2 <= size <= 4:
        return 1      # Small
    else:
        return 2      # Large

df2['FamilyGroup'] = df2['FamilySize'].apply(family_label)

# -----------------------------
# 7. Fare 변환
# -----------------------------
df2['Log_Fare'] = np.log1p(df2['Fare'])

# Fare 구간화
df2['Fare_Group'] = pd.qcut(df2['Fare'], q=4, labels=False, duplicates='drop')
df2['TicketGroup'] = df2.groupby('Ticket')['Ticket'].transform('count')
# -----------------------------
# 8. 불필요 컬럼 제거
# -----------------------------
df2 = df2.drop(['Name', 'LastName', 'Ticket', 'Cabin', 'Age', 'FamilyGroup'], axis=1)


# -----------------------------
# 9. 최종 확인
# -----------------------------
print(df2.head())
print(df2.info())
print(df2.isnull().sum())
df2

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name  Sex   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris    0  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1  38.0      1      0   
2                             Heikkinen, Miss. Laina    1  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1  35.0      1      0   
4                           Allen, Mr. William Henry    0  35.0      0      0   

             Ticket     Fare  ... FamilyName_Count  Title  Age_Group  \
0         A/5 21171   7.2500  ...                2      0          2   
1          PC 17599  71.2833  ...                1      2          3   
2  STON/O2. 3101282   7.9250  ...                1      1          2   
3       

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,FamilyName_Count,Title,Age_Group,Age_scaled,FamilySize,IsAlone,FamilyGroup,Log_Fare,Fare_Group,TicketGroup
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,...,2,0,2,-0.565736,2,0,1,2.110213,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,...,1,2,3,0.663861,2,0,1,4.280593,3,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,...,1,1,2,-0.258337,1,1,0,2.188856,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,...,2,2,2,0.433312,2,0,1,3.990834,3,2
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,...,2,0,2,0.433312,1,1,0,2.202765,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",0,27.0,0,0,211536,13.0000,...,1,4,2,-0.181487,1,1,0,2.639057,1,1
887,888,1,1,"Graham, Miss. Margaret Edith",1,19.0,0,0,112053,30.0000,...,3,1,1,-0.796286,1,1,0,3.433987,2,1
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",1,28.0,1,2,W./C. 6607,23.4500,...,2,1,2,-0.104637,4,0,1,3.196630,2,2
889,890,1,1,"Behr, Mr. Karl Howell",0,26.0,0,0,111369,30.0000,...,1,0,2,-0.258337,1,1,0,3.433987,2,1


In [9]:
df2 = df2.drop('FamilyGroup',axis=1)

In [43]:
X = df2.drop('Survived', axis=1)
y = df2['Survived']
X_train, X_test, y_train, y_test = train_test_split( X, y, stratify=y, random_state=43)
yujin = DecisionTreeClassifier( max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42)
yujin.fit( X_train , y_train )

DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, min_samples_split=20,
                       random_state=42)

In [44]:
y_pred = yujin.predict( X_test )

print("--- 오차 행렬 ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- 분류 성능 평가 ---")
print(f"정확도 (Accuracy): {accuracy_score(y_test, y_pred):.2%}")
print(f"정밀도 (Precision): {precision_score(y_test, y_pred):.2%}")
print(f"재현율 (Recall)  : {recall_score(y_test, y_pred):.2%}")
print(f"F1 점수 (F1 Score): {f1_score(y_test, y_pred):.2%}")

--- 오차 행렬 ---
[[129   8]
 [ 26  60]]

--- 분류 성능 평가 ---
정확도 (Accuracy): 84.75%
정밀도 (Precision): 88.24%
재현율 (Recall)  : 69.77%
F1 점수 (F1 Score): 77.92%


In [7]:
scale_cols = ['Log_Fare', 'FamilySize', 'Parch', 'SibSp']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

In [8]:
#KNN
얼리2 = KNeighborsClassifier(n_neighbors=5)
얼리2.fit(X_train_scaled, y_train)
y_pred2 = 얼리2.predict( X_test_scaled )

print("--- 오차 행렬 ---")
print(confusion_matrix(y_test, y_pred2))

print("\n--- 분류 성능 평가 ---")
print(f"정확도 (Accuracy): {accuracy_score(y_test, y_pred2):.2%}")
print(f"정밀도 (Precision): {precision_score(y_test, y_pred2):.2%}")
print(f"재현율 (Recall)  : {recall_score(y_test, y_pred2):.2%}")
print(f"F1 점수 (F1 Score): {f1_score(y_test, y_pred2):.2%}")

--- 오차 행렬 ---
[[105  32]
 [ 66  20]]

--- 분류 성능 평가 ---
정확도 (Accuracy): 56.05%
정밀도 (Precision): 38.46%
재현율 (Recall)  : 23.26%
F1 점수 (F1 Score): 28.99%


In [36]:
#Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=7, random_state=24)
rf.fit( X_train , y_train )
y_pred3 = rf.predict( X_test )

print("--- 오차 행렬 ---")
print(confusion_matrix(y_test, y_pred3))

print("\n--- 분류 성능 평가 ---")
print(f"정확도 (Accuracy): {accuracy_score(y_test, y_pred3):.2%}")
print(f"정밀도 (Precision): {precision_score(y_test, y_pred3):.2%}")
print(f"재현율 (Recall)  : {recall_score(y_test, y_pred3):.2%}")
print(f"F1 점수 (F1 Score): {f1_score(y_test, y_pred3):.2%}")

--- 오차 행렬 ---
[[121  16]
 [ 19  67]]

--- 분류 성능 평가 ---
정확도 (Accuracy): 84.30%
정밀도 (Precision): 80.72%
재현율 (Recall)  : 77.91%
F1 점수 (F1 Score): 79.29%
